# Kimodo: EmbeddingGemma 2 → 4096차원 어댑터 (Colab T4)
런타임 유형에서 **T4 GPU**를 선택하고, 허가된 학습 전용 `kimodo-train-input.zip`을 Colab 파일 패널로 `/content`에 올린 뒤 **모두 실행**을 한 번 누른다. 기본 모드는 `train`이다. 입력 검증, 환경 준비, 학생 임베딩, 학습, 결과 ZIP 저장이 중간 확인 없이 이어진다. 입력이 누락·손상되거나 CPU 런타임이면 중단한다. 최종 시험 데이터와 Llama 가중치는 Colab에 올리지 않는다.
선택 사항 `smoke`는 학습 데이터 없이 쓰는 합성 타깃 진단이다. 별도로 먼저 실행할 필요는 없다. T4에는 네이티브 BF16이 없고 EmbeddingGemma 2는 FP16을 지원하지 않으므로 이 노트북은 FP32 파일럿이다. v1 BF16 결과 및 RTX 4090 자원 평가를 대신하지 않는다.

In [ ]:
from pathlib import Path

MODE = 'train'  # 학습 입력 없이 설치만 진단하려면 'smoke'
INPUT_PATH = Path('/content/kimodo-train-input.zip')  # 교사 캐시 도구가 만든 ZIP 하나
OUT_DIR = Path('/content/kimodo-runs') / ('smoke-01' if MODE == 'smoke' else 't4-fp32-pilot-01')
EPOCHS, ADAPTER_BATCH, ENCODER_BATCH, LR, SEED = 12, 64, 24, 0.002, 17
MODEL_ID = 'google/embeddinggemma-2'
MODEL_REVISION = 'f9972ac22e696a105c482202646979e33e4b5a1e'
DEPENDENCIES = ('transformers==5.24.0', 'sentence-transformers==6.3.0')
ENV_DIR = Path('/content/kimodo-gemma2-venv')
PROMPT_NAME = 'SentenceSimilarity'  # 양 언어에 같은 공식 task prefix
MAX_SEQ_LENGTH = 512


In [ ]:
import hashlib
import json
from importlib.metadata import version
import inspect
import math
import os
from pathlib import Path
import shutil
import subprocess
import sys
import tempfile
from zipfile import ZipFile

import numpy as np
import torch
from safetensors.torch import save_file

def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

def load_training_bundle(root):
    manifest, targets, provenance = (Path(root) / name for name in ('train.jsonl', 'targets.npz', 'teacher.json'))
    missing = [p.name for p in (manifest, targets, provenance) if not p.is_file()]
    if missing:
        raise FileNotFoundError(f'{root}: missing input files: {", ".join(missing)}')
    meta = json.loads(provenance.read_text(encoding='utf-8'))
    if meta.get('schema') != 'kimodo.teacher.v1' or meta.get('teacher') != 'kimodo-llm2vec':
        raise ValueError('Expected a kimodo.teacher.v1 LLM2Vec provenance file')
    for path, key in ((manifest, 'manifest_sha256'), (targets, 'targets_sha256')):
        if sha256(path) != meta.get(key):
            raise ValueError(f'{path.name} SHA-256 does not match teacher.json')
    rows = []
    seen = set()
    with manifest.open(encoding='utf-8') as stream:
        for number, line in enumerate(stream, 1):
            row = json.loads(line)
            if not isinstance(row, dict) or row.get('split') != 'train':
                raise ValueError(f'Line {number}: only train rows are allowed in Colab')
            if not all(isinstance(row.get(k), str) and row[k].strip() for k in ('id', 'group_id', 'en', 'ko')):
                raise ValueError(f'Line {number}: missing id, group_id, en, or ko')
            if row['id'] in seen:
                raise ValueError(f'Line {number}: duplicate id')
            if not all(len(row[k]) <= 600 for k in ('en', 'ko')):
                raise ValueError(f'Line {number}: review text over 600 characters before training')
            seen.add(row['id'])
            rows.append(row)
            if len(rows) > 65536:
                raise ValueError('This Colab pilot supports at most 65536 training pairs per run')
    if not rows:
        raise ValueError('Empty training manifest')
    if meta.get('pairs') != len(rows):
        raise ValueError('Record count does not match teacher.json')
    with np.load(targets, allow_pickle=False) as cache:
        ids, vectors = cache['ids'], cache['vectors']
    if ids.shape != (len(rows),) or ids.dtype.kind != 'U' or len(set(ids.tolist())) != len(rows):
        raise ValueError('Cache ids must contain exactly one unique Unicode id per pair')
    if set(ids.tolist()) != seen:
        raise ValueError('Cache ids do not match train.jsonl')
    if vectors.shape != (len(rows), 4096) or vectors.dtype != np.float32 or not np.isfinite(vectors).all():
        raise ValueError('Teacher cache must have finite float32 vectors of shape [pairs, 4096]')
    order = {key: i for i, key in enumerate(ids.tolist())}
    return rows, vectors[[order[r['id']] for r in rows]], meta

def load_training_input(path):
    path = Path(path)
    if path.is_dir():
        return load_training_bundle(path)
    if not path.is_file():
        raise FileNotFoundError(f'Upload the teacher cache ZIP to {path} before Run all')
    limits_mib = {'train.jsonl': 256, 'targets.npz': 1200, 'teacher.json': 1}
    with ZipFile(path) as archive, tempfile.TemporaryDirectory() as tmp:
        members = archive.infolist()
        if len(members) != len(limits_mib) or {m.filename for m in members} != set(limits_mib):
            raise ValueError('Input ZIP must contain exactly train.jsonl, targets.npz, and teacher.json at its root')
        for member in members:
            if member.file_size > limits_mib[member.filename] * 1024 * 1024:
                raise ValueError(f'Input ZIP member is too large: {member.filename}')
            with archive.open(member) as source, (Path(tmp) / member.filename).open('wb') as dest:
                shutil.copyfileobj(source, dest)
        return load_training_bundle(Path(tmp))

def fit_adapter(features, targets, *, device, epochs, batch_size, lr, seed):
    if targets.ndim != 2 or targets.shape[1] != 4096 or features.shape != (2, len(targets), 768) or not len(targets):
        raise ValueError('Expected features [2, pairs, 768] and raw targets [pairs, 4096]')
    if not all(np.isfinite(a).all() for a in (features, targets)) or epochs < 1 or batch_size < 1 or not math.isfinite(lr) or lr <= 0:
        raise ValueError('Non-finite data or invalid optimizer parameters')
    x = torch.as_tensor(features, dtype=torch.float32, device='cpu').flatten(0, 1)
    y = torch.as_tensor(targets, dtype=torch.float32, device='cpu')
    n = len(y)
    random = torch.Generator(device='cpu').manual_seed(seed)
    with torch.random.fork_rng(devices=[]):
        torch.random.default_generator.manual_seed(seed)
        model = torch.nn.Linear(768, 4096, device='cpu', dtype=torch.float32).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0)

    @torch.inference_mode()
    def stats():
        report = {}
        for arm, offset in (('en', 0), ('ko', n)):
            mse = cosine = norm_error = target_norm = 0.0
            for start in range(0, n, batch_size):
                stop = min(start + batch_size, n)
                gold = y[start:stop].to(device)
                pred = model(x[offset + start:offset + stop].to(device))
                mse += (pred - gold).square().mean(dim=-1).sum().item()
                cosine += torch.nn.functional.cosine_similarity(pred, gold).sum().item()
                norms = gold.norm(dim=-1)
                norm_error += (pred.norm(dim=-1) - norms).abs().sum().item()
                target_norm += norms.sum().item()
            report[arm] = {'mse': mse / n, 'cosine': cosine / n, 'norm_mae': norm_error / n, 'target_norm_mean': target_norm / n}
        return report

    before = stats()
    history = []
    for epoch in range(epochs):
        epoch_loss = 0.0
        for indices in torch.randperm(2 * n, generator=random).split(batch_size):
            optimizer.zero_grad(set_to_none=True)
            prediction = model(x[indices].to(device))
            loss = torch.nn.functional.mse_loss(prediction, y[indices % n].to(device))
            if not torch.isfinite(loss):
                raise RuntimeError('Non-finite adapter loss')
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item() * len(indices)
        history.append(epoch_loss / (2 * n))
        print(f'epoch {epoch + 1}/{epochs}: train MSE {history[-1]:.6f}', flush=True)
    return {'state': {k: v.detach().cpu().contiguous() for k, v in model.state_dict().items()}, 'before': before, 'metrics': stats(), 'epoch_mse': history}

def save_result(out, result, provenance):
    out = Path(out)
    out.mkdir(parents=True, exist_ok=False)
    checkpoint = out / 'adapter.safetensors'
    save_file(result['state'], str(checkpoint))
    identity = {**provenance, 'checkpoint_sha256': sha256(checkpoint)}
    material = json.dumps(identity, ensure_ascii=False, sort_keys=True, allow_nan=False).encode('utf-8')
    report = {**identity, 'run_id': hashlib.sha256(material).hexdigest(),
              'train_diagnostics_before': result['before'], 'train_diagnostics_after': result['metrics'], 'epoch_mse': result['epoch_mse']}
    metadata = out / 'run.json'
    metadata.write_text(json.dumps(report, ensure_ascii=False, indent=2, allow_nan=False) + '\n', encoding='utf-8')
    pending = out / 'kimodo-result.zip.tmp'
    with ZipFile(pending, 'x') as archive:
        for file in (checkpoint, metadata):
            archive.write(file, file.name)
    pending.replace(out / 'kimodo-result.zip')
    return report


In [ ]:
# 별도 프로세스에서 임베딩만 만들고 종료해 GPU 메모리를 반환한다.
ENCODER_SOURCE = r'''
import json, os, sys
from importlib.metadata import version
from pathlib import Path
import numpy as np
import torch
from sentence_transformers import SentenceTransformer
from huggingface_hub import snapshot_download

root = Path(sys.argv[1])
c = json.loads((root / 'encode.json').read_text(encoding='utf-8'))
assert torch.cuda.is_available() and 'T4' in torch.cuda.get_device_name(0)
torch.cuda.reset_peak_memory_stats()
model_path = snapshot_download(c['model'], revision=c['revision'],
    allow_patterns=['*.json', 'model.safetensors', 'tokenizer.model', 'chat_template.jinja'])
model = SentenceTransformer(model_path, device='cuda:0', trust_remote_code=False,
    config_kwargs={'vision_config': None, 'audio_config': None},
    model_kwargs={'dtype': torch.float32, 'attn_implementation': 'eager'})
model.eval().requires_grad_(False)
backbone = model[0].auto_model
if backbone.vision_tower is not None or backbone.audio_tower is not None:
    raise RuntimeError('Expected the text-only Gemma configuration')
if {p.dtype for p in model.parameters() if p.is_floating_point()} != {torch.float32}:
    raise RuntimeError('T4 pilot requires an FP32 encoder')
model.max_seq_length = c['max_seq_length']
if model.get_embedding_dimension() != 768:
    raise RuntimeError('Expected native 768d text embeddings')
prefix = model.prompts[c['prompt_name']]
# Same task prefix for English and Korean; no silent truncation.
out = []
for i in range(0, len(c['texts']), c['batch']):
    batch = c['texts'][i:i + c['batch']]
    tokens = model.tokenizer([prefix + text for text in batch], truncation=False)['input_ids']
    if any(len(x) > c['max_seq_length'] for x in tokens):
        raise ValueError(f'Text too long for recorded token limit, batch offset {i}')
    vec = model.encode(batch, batch_size=c['batch'], prompt_name=c['prompt_name'],
        normalize_embeddings=True, convert_to_numpy=True, show_progress_bar=False)
    out.append(np.asarray(vec, dtype=np.float32))
features = np.concatenate(out)
if features.shape != (len(c['texts']), 768) or not np.isfinite(features).all():
    raise RuntimeError('Unexpected or invalid student embedding')
np.save(root / 'features.npy', features, allow_pickle=False)
stats = {'dtype': 'float32', 'prompt_name': c['prompt_name'], 'prompt_prefix': prefix,
    'pooling': 'checkpoint mean pooling + built-in projection + L2 normalization',
    'max_seq_length': c['max_seq_length'], 'batch_size': c['batch'], 'gpu': torch.cuda.get_device_name(0),
    'gpu_capability': list(torch.cuda.get_device_capability(0)),
    'encoder_peak_torch_allocated_bytes': torch.cuda.max_memory_allocated(),
    'versions': {p: version(p) for p in ('torch', 'transformers', 'sentence-transformers', 'huggingface-hub')}}
(root / 'encoder.json').write_text(json.dumps(stats), encoding='utf-8')
print(f'Encoded {len(features)} texts on {stats["gpu"]}.', flush=True)
'''

def runtime():
    if not torch.cuda.is_available() or 'T4' not in torch.cuda.get_device_name(0):
        raise RuntimeError('Connect a Colab Tesla T4 GPU before running the notebook')
    subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv'], check=True)
    exe = ENV_DIR / 'bin/python'
    if not exe.is_file():
        subprocess.run([sys.executable, '-m', 'venv', '--without-pip', '--system-site-packages', str(ENV_DIR)], check=True)
    subprocess.run([sys.executable, '-m', 'pip', '--python', str(exe), 'install', *DEPENDENCIES], check=True)
    return exe

def encode_rows(rows, exe):
    with tempfile.TemporaryDirectory() as tmp:
        root = Path(tmp)
        script = root / 'encode.py'
        script.write_text(ENCODER_SOURCE, encoding='utf-8')
        config = {'model': MODEL_ID, 'revision': MODEL_REVISION, 'batch': ENCODER_BATCH,
                  'prompt_name': PROMPT_NAME, 'max_seq_length': MAX_SEQ_LENGTH,
                  'texts': [row[lang].strip() for lang in ('en', 'ko') for row in rows]}
        (root / 'encode.json').write_text(json.dumps(config, ensure_ascii=False), encoding='utf-8')
        subprocess.run([str(exe), str(script), str(root)], check=True,
                       env={**os.environ, 'TOKENIZERS_PARALLELISM': 'false', 'HF_HUB_DISABLE_PROGRESS_BARS': '1'})
        stats = json.loads((root / 'encoder.json').read_text(encoding='utf-8'))
        vectors = np.load(root / 'features.npy', allow_pickle=False).reshape(2, len(rows), 768)
    return vectors, stats


In [ ]:
def smoke_input():
    pairs = [('Raise the left hand.', '왼손을 든다.'), ('Raise the right hand.', '오른손을 든다.'),
             ('Turn left and then crouch.', '왼쪽으로 돈 뒤 웅크린다.'), ('Stand up from a low pose.', '낮은 자세에서 일어난다.')]
    rows = [{'id': f'smoke-{i}', 'group_id': 'smoke', 'split': 'train', 'en': en, 'ko': ko} for i, (en, ko) in enumerate(pairs)]
    rng = np.random.default_rng(SEED)
    targets = rng.standard_normal((len(rows), 4096), dtype=np.float32)
    meta = {'teacher': 'synthetic-smoke-only', 'manifest_sha256': hashlib.sha256(json.dumps(rows, sort_keys=True).encode()).hexdigest(),
            'targets_sha256': hashlib.sha256(targets.tobytes()).hexdigest()}
    return rows, targets, meta

def main():
    if MODE not in ('smoke', 'train'):
        raise ValueError('MODE must be smoke or train')
    if OUT_DIR.exists():
        raise FileExistsError(f'Change OUT_DIR before re-running: {OUT_DIR}')
    print('[1/5] Validate training input', flush=True)
    rows, targets, teacher = smoke_input() if MODE == 'smoke' else load_training_input(INPUT_PATH)
    print(f'Validated {len(rows)} pairs; teacher: {teacher["teacher"]}', flush=True)
    print('[2/5] Validate GPU and prepare isolated encoder environment', flush=True)
    exe = runtime()
    print('[3/5] Compute English and Korean embeddings', flush=True)
    features, encoder = encode_rows(rows, exe)
    print('[4/5] Fit the adapter', flush=True)
    torch.cuda.reset_peak_memory_stats()
    result = fit_adapter(features, targets, device='cuda:0', epochs=EPOCHS, batch_size=ADAPTER_BATCH, lr=LR, seed=SEED)
    if MODE == 'smoke' and not all(result['metrics'][a]['mse'] < result['before'][a]['mse'] for a in ('en', 'ko')):
        raise RuntimeError('Smoke test: training MSE did not drop for both languages')
    print('[5/5] Save checkpoint, report, and result ZIP', flush=True)
    code = ENCODER_SOURCE + ''.join(inspect.getsource(f) for f in (sha256, load_training_bundle, load_training_input, fit_adapter, save_result, runtime, encode_rows, smoke_input, main))
    provenance = {'mode': MODE, 'protocol': 't4-fp32-pilot-not-v1', 'student': MODEL_ID, 'student_revision': MODEL_REVISION,
                  'encoder': encoder, 'teacher': teacher, 'pairs': len(rows), 'seed': SEED, 'lr': LR,
                  'epochs': EPOCHS, 'adapter_batch_size': ADAPTER_BATCH, 'adapter_dtype': 'float32',
                  'adapter_peak_torch_allocated_bytes': torch.cuda.max_memory_allocated(),
                  'notebook_kernel_versions': {p: version(p) for p in ('numpy', 'torch', 'safetensors')},
                  'code_sha256': hashlib.sha256(code.encode('utf-8')).hexdigest()}
    report = save_result(OUT_DIR, result, provenance)
    print(json.dumps({'result_zip': str(OUT_DIR / 'kimodo-result.zip'), 'run_id': report['run_id'], 'after': report['train_diagnostics_after'],
                      'encoder_peak_torch_allocated_bytes': encoder['encoder_peak_torch_allocated_bytes'],
                      'adapter_peak_torch_allocated_bytes': provenance['adapter_peak_torch_allocated_bytes']}, indent=2))

main()


## 산출물
완료 후 Colab 파일 패널에서 `OUT_DIR/kimodo-result.zip` 하나를 다운로드한다. ZIP에는 `adapter.safetensors`와 `run.json`만 있다. 합성 smoke 체크포인트는 배포하지 않는다. Google Drive는 자동으로 연결하지 않으므로 런타임을 닫기 전에 결과를 다운로드한다. 학습 메모리는 PyTorch 프로세스 할당량이며 v1의 RTX 4090 전체 추론 메모리 지표와 다르다.